# Avito: кандидатогенерация — v3, трудная валидация и ранкер

Продолжение `01_eda_baseline.ipynb` (тег `v2`: Recall@50 = **0,8313** на лидерборде при 0,877 на валидации).

**Что меняется в v3**
1. **Валидация, согласованная с лидербордом.** Корпус бенчмарка, по-видимому, собран вокруг его запросов, поэтому у настоящих позитивов много похожих конкурентов, а у подмешанных позитивов валидации v2 — мало. Сравниваются две схемы: прежняя (`injected`) и запросы train, чьи выбранные объявления уже лежат в корпусе бенчмарка (`in_corpus`). Берётся та, где веса v2 ближе всего к результату на лидерборде.
2. **Новые признаки и генератор:** символьные n-граммы (опечатки), поле «услуга» без адреса, совпадение запроса с адресом («город в запросе»), признаки контекста запроса, ранги кандидата в каждом списке пула.
3. **Ранкер LightGBM** вместо линейной формулы. Обучается на фолдах запросов из train; признаки каждого фолда считаются по train без его строк.

| Раздел | Что происходит |
|---|---|
| 0. Окружение | код из репозитория, зависимости, сиды |
| 1. Данные | загрузка, сегменты запросов |
| 2. Выборки валидации | две схемы, стратифицированные под бенчмарк |
| 3. Выбор схемы валидации | пулы, калибровка по лидерборду, проверка воспроизведения v2 |
| 4. Фолды для ранкера | выборки запросов и обучающие строки |
| 5. Обучение ранкера | LightGBM, ранняя остановка по Recall@50 |
| 6. Качество на валидации | сравнение с формулой, сегменты, важность признаков |
| 7. Бенчмарк | предсказание, `answer.csv`, проверка формата |
| 8. Артефакты | модель, отчёт, md5 |

Полный прогон на CPU Kaggle — около 45 минут, нужно ~16 ГБ RAM. Для быстрой проверки кода: переменная окружения `SMOKE_TEST=1` уменьшает выборки.

## 0. Окружение

Ячейка находит код решения: локально — папку репозитория рядом с ноутбуком; на Kaggle — клонирует репозиторий вместе с тегами (токен берётся из Kaggle Secrets и не попадает ни в вывод, ни на диск). `REPO_REF` — ветка, тег отправленной версии или хеш коммита. Число потоков BLAS фиксируется **до** импорта numpy.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path

N_THREADS = 4
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = str(N_THREADS)

IS_KAGGLE = Path("/kaggle/input").exists()
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"          # ветка, тег (например, "v2") или хеш коммита
REPO_DIR = Path("/kaggle/working/avito-candgen")


def github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def checkout_repo() -> Path:
    """Клонирует (или обновляет) репозиторий вместе с тегами и переключается на REPO_REF."""
    token = github_token()
    if not REPO_DIR.exists():
        git("clone", "--quiet", REPO_URL, str(REPO_DIR), token=token)
    git("-C", str(REPO_DIR), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    git("-C", str(REPO_DIR), "checkout", "--quiet", "--force", "--detach",
        f"origin/{REPO_REF}" if is_branch else REPO_REF)
    print("commit:", git("-C", str(REPO_DIR), "rev-parse", "HEAD"))
    return REPO_DIR


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:          # локальный запуск
        if (path / "src" / "pipeline.py").exists():
            return path
    if not IS_KAGGLE:
        raise RuntimeError("Не найден код решения (src/). Запустите ноутбук внутри репозитория.")
    root = checkout_repo()
    if not (root / "src" / "pipeline.py").exists():
        raise RuntimeError(f"В корне репозитория нет src/pipeline.py: {sorted(p.name for p in root.iterdir())}")
    return root


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

# пакеты, которых может не быть в образе (версии — как в requirements.txt)
for module, packages in (("pymorphy3", ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"]),
                         ("lightgbm", ["lightgbm==4.6.0"])):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("repo:", REPO_ROOT, "| kaggle:", IS_KAGGLE)

In [ ]:
import json
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from src import analysis, eda
from src.candidates import BASE_FEATURES, build_vocabs
from src.config import CFG, RANKER_CFG
from src.data import load_benchmark, load_train, load_train_items_text
from src.paths import get_data_dir, get_output_dir, get_work_dir
from src.pipeline import STATS_COLS, ItemLemmaCache, add_lemma_keys, build_index, build_pool
from src.ranker import (add_stage1, importance_table, ranker_score, sample_training_rows, train_ranker)
from src.ranking import PoolData, linear_score, predict_from_scores, weights_vector
from src.repro import library_versions, seed_everything
from src.sampling import bench_targets, group_table, sample_queries
from src.submit import save_answer, validate_answer
from src.text import Lemmatizer
from src.utils import timer
from src.validation import add_query_segments, mark_seen, per_query_recall, recall_at_k

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE_TEST") == "1"
R = replace(RANKER_CFG, n_val_queries=300, fold_queries=300, max_rounds=150, early_stopping=30) if SMOKE else RANKER_CFG
DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
K, DEC = CFG.top_k, CFG.score_decimals
V2_W = weights_vector(BASE_FEATURES, R.v2_weights)
print(f"версия решения: {R.version}{' (SMOKE_TEST)' if SMOKE else ''}\ndata: {DATA_DIR}\nwork: {WORK_DIR}\nout:  {OUT_DIR}")
print(VERSIONS)

## 1. Данные

Загрузка и подготовка — как в v2: id приводятся к строкам, у запросов считаются «мешок лемм» и сегменты (фильтр, тип локации, знакомый ли текст). Дополнительно для каждой группы-запроса отмечается, лежат ли все её выбранные объявления в корпусе бенчмарка.

In [ ]:
with timer("загрузка"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])

OVERLAP = eda.overlap(train, bench_q, bench_items, CFG.item_stats_min_overlap)
print(f"\nгрупп-запросов: {len(groups):,}; из них все выбранные объявления в корпусе: "
      f"{groups['in_corpus'].sum():,} ({groups['in_corpus'].mean():.3f})")

## 2. Выборки валидации

Обе схемы повторяют состав бенчмарка (знакомый/новый текст × фильтр × тип локации) и используют одинаковую соль, то есть одни и те же убранные из train тексты:
* `injected` — любые запросы; их объявления, которых нет в корпусе, подмешиваются в копию корпуса (как в v2);
* `in_corpus` — только запросы, чьи объявления уже лежат в корпусе бенчмарка, со своим настоящим окружением.

«Новые» запросы теперь берутся по одному на текст: новые тексты бенчмарка почти всегда редкие, а выбор по группам перекашивал валидацию v2 в сторону популярных текстов.

In [ ]:
ALL_ROWS = np.ones(len(train), dtype=bool)
KEYS = {"injected": pd.Index(groups["query_key"]),
        "in_corpus": pd.Index(groups.loc[groups["in_corpus"], "query_key"])}
val_target = bench_targets(bench_q, R.n_val_queries)

with timer("выборки валидации"):
    VAL = {name: sample_queries(f"валидация {name}", train, groups, val_target, ALL_ROWS, keys,
                                R.text_holdout_frac, R.val_salt)
           for name, keys in KEYS.items()}

pd.concat({name: s.report.set_index(["текст", "страта"])["факт"] for name, s in VAL.items()},
          axis=1).assign(цель=val_target.to_numpy())

## 3. Выбор схемы валидации

Строятся два индекса с расширенными признаками: корпус бенчмарка как есть (для `in_corpus` и для предсказания) и его копия с подмешанными позитивами (для `injected`). Затем — пулы для обеих схем и для бенчмарка.

In [ ]:
vocabs = build_vocabs([train, bench_q], [train, bench_items])
cache = ItemLemmaCache(lem, CFG.desc_max_chars)
bench_ids = frozenset(bench_items["item_id"])       # только проверка вхождения


def items_with_injected(samples) -> pd.DataFrame:
    """Корпус бенчмарка + эталонные объявления выборок, которых в нём нет."""
    missing = sorted(dict.fromkeys(i for s in samples for rel in s.truth for i in rel if i not in bench_ids))
    extra = load_train_items_text(DATA_DIR, missing)
    print(f"подмешано объявлений: {len(extra):,}")
    return pd.concat([bench_items, extra[bench_items.columns]], ignore_index=True)


def make_pool(name, queries, truth, index, items, stats_mask):
    """Пул v3 с признаками и скором формулы v2 (stage1)."""
    _, pool = build_pool(name, index, items, queries, train.loc[stats_mask, STATS_COLS], lem=lem,
                         vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"],
                         truth=truth, ext_cfg=R)
    return add_stage1(pool, index, R.v2_weights, DEC)


bench_index = build_index("корпус бенчмарка", bench_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
inj_items = items_with_injected([VAL["injected"]])
inj_index = build_index("корпус injected", inj_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
CORPORA = {"injected": (inj_index, inj_items), "in_corpus": (bench_index, bench_items)}

VAL_POOLS = {name: make_pool(f"валидация {name}", s.queries, s.truth, *CORPORA[name], s.stats_mask)
             for name, s in VAL.items()}
BENCH_POOL = make_pool("бенчмарк", bench_q, None, bench_index, bench_items, ALL_ROWS)

**Калибровка.** Для каждой схемы считается Recall@50 ровно той модели, что была отправлена (пул v2 + веса v2), и сравнивается с лидербордом. Выбирается схема с наименьшим расхождением. Таблица «трудности» показывает, насколько плотная конкуренция у запросов каждого набора по сравнению с бенчмарком.

In [ ]:
calibration = []
for name, s in VAL.items():
    pool, (index, _) = VAL_POOLS[name], CORPORA[name]
    v2_pool = analysis.v2_rows(pool)
    recall_v2 = PoolData(v2_pool, index, BASE_FEATURES, s.n_rel).recall(V2_W, K, DEC)
    calibration.append({
        "схема": name, "запросов": len(s.queries), "Recall@50 v2": recall_v2,
        "LB v2": R.v2_lb, "расхождение": recall_v2 - R.v2_lb,
        "полнота пула v2": analysis.pool_hit_rate(v2_pool, s.n_rel).mean(),
        "полнота пула v3": analysis.pool_hit_rate(pool, s.n_rel).mean(),
    })
calibration = pd.DataFrame(calibration).set_index("схема").round(4)
SCHEME = calibration["расхождение"].abs().idxmin() if R.val_scheme == "auto" else R.val_scheme
display(calibration)
print(f"→ схема валидации: {SCHEME}")

display(analysis.difficulty_table({"бенчмарк": BENCH_POOL, **{f"валидация {n}": p for n, p in VAL_POOLS.items()}}, K))

In [ ]:
# Проверка: пайплайн v3 без новых кандидатов и с весами v2 даёт ровно отправленный ответ v2
v2_bench = analysis.v2_rows(BENCH_POOL)
v2_pred = predict_from_scores(v2_bench, bench_index, linear_score(v2_bench[BASE_FEATURES].to_numpy(np.float64), V2_W),
                              K, DEC, len(bench_q))
v2_md5 = save_answer(bench_q["query_id"], v2_pred, WORK_DIR / "answer_v2_check.csv")
print(f"md5 ответа v2 из пайплайна v3: {v2_md5} | отправленный: {R.v2_answer_md5} | "
      f"совпадает: {v2_md5 == R.v2_answer_md5}")

## 4. Фолды для ранкера

Запросы для обучения берутся по той же схеме и в тех же пропорциях, что и валидация, но не пересекаются с ней: фолды строятся только из строк train, доступных валидационным статистикам. Для каждого фолда статистики считаются без его собственных строк — иначе признаки вроде P(микрокатегория | запрос) «подсказывали» бы ответ.

Первые `n_folds − 1` фолдов идут в обучение (все позитивы + трудные, символьно похожие и случайные негативы), последний — для ранней остановки (полный пул).

In [ ]:
eligible = KEYS[SCHEME].difference(pd.Index(VAL[SCHEME].keys))
fold_target = bench_targets(bench_q, R.fold_queries)
FOLDS = []
with timer("выборки фолдов"):
    for f in range(R.n_folds):
        fold = sample_queries(f"фолд {f}", train, groups, fold_target, VAL[SCHEME].stats_mask, eligible,
                              R.fold_holdout_frac, f"{R.val_salt}-fold{f}")
        eligible = eligible.difference(pd.Index(fold.keys))
        FOLDS.append(fold)
print("запросов в фолдах:", [len(f.queries) for f in FOLDS])

if SCHEME == "injected":        # позитивы фолдов тоже нужно подмешать; валидацию пересчитываем на том же корпусе
    fold_items = items_with_injected([VAL["injected"], *FOLDS])
    fold_index = build_index("корпус injected + фолды", fold_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
    CORPORA["injected"] = (fold_index, fold_items)
    VAL_POOLS["injected"] = make_pool("валидация injected", VAL["injected"].queries, VAL["injected"].truth,
                                      fold_index, fold_items, VAL["injected"].stats_mask)
else:
    fold_index, fold_items = bench_index, bench_items

In [ ]:
train_parts = []
for f, fold in enumerate(FOLDS):
    pool = make_pool(f"фолд {f}", fold.queries, fold.truth, fold_index, fold_items, fold.stats_mask)
    if f < R.n_folds - 1:
        rows = sample_training_rows(pool, R, seed=CFG.seed + f)
        rows.insert(0, "gid", f * 1_000_000 + rows["q"].astype(np.int64))
        train_parts.append(rows)
    else:
        VALID_POOL = pool
    del pool

TRAIN_ROWS = pd.concat(train_parts, ignore_index=True)
del train_parts
print(f"обучающих строк: {len(TRAIN_ROWS):,} (позитивов {int(TRAIN_ROWS['label'].sum()):,}); "
      f"строк в фолде остановки: {len(VALID_POOL):,}")

## 5. Обучение ранкера

Сравниваются две целевые функции LightGBM — `lambdarank` и `binary`. Ранняя остановка и выбор между ними — по Recall@50 на полном пуле последнего фолда; валидация из раздела 2 в обучении не участвует.

In [ ]:
valid_fold = FOLDS[-1]
valid_rank = fold_index.rank[VALID_POOL["item"].to_numpy()]
fold_scores = {"формула v2": PoolData(VALID_POOL, fold_index, BASE_FEATURES, valid_fold.n_rel).recall(V2_W, K, DEC)}
MODELS = {}
for objective in R.objectives:
    with timer(f"LightGBM {objective}"):
        booster, best_iter, best = train_ranker(TRAIN_ROWS, VALID_POOL, valid_fold.n_rel, valid_rank,
                                                objective, R, CFG.seed, N_THREADS, K, DEC)
    MODELS[objective] = booster
    fold_scores[f"ранкер {objective}"] = best
    print(f"{objective}: лучшая итерация {best_iter}, Recall@{K} на фолде {best:.4f}")

OBJECTIVE = max(R.objectives, key=lambda o: fold_scores[f"ранкер {o}"])
RANKER = MODELS[OBJECTIVE]
pd.Series(fold_scores, name=f"Recall@{K} на фолде остановки").round(4).to_frame()

## 6. Качество на валидации

Валидация не участвовала ни в обучении, ни в ранней остановке, поэтому это честная оценка. Метрика считается «как на платформе» — по спискам `item_id`.

In [ ]:
val_s, val_pool = VAL[SCHEME], VAL_POOLS[SCHEME]
val_index = CORPORA[SCHEME][0]
nq = len(val_s.queries)


def val_recall(pool, score):
    return recall_at_k(predict_from_scores(pool, val_index, score, K, DEC, nq), val_s.truth, K)


v2_pool = analysis.v2_rows(val_pool)
ranker_val_score = ranker_score(RANKER, val_pool, N_THREADS)
VAL_RESULTS = {
    "v2: пул v2 + формула v2": val_recall(v2_pool, v2_pool["stage1"].to_numpy(np.float64)),
    "формула v2 на пуле v3": val_recall(val_pool, val_pool["stage1"].to_numpy(np.float64)),
    f"ранкер ({OBJECTIVE})": val_recall(val_pool, ranker_val_score),
}
USE_RANKER = VAL_RESULTS[f"ранкер ({OBJECTIVE})"] > VAL_RESULTS["формула v2 на пуле v3"]
display(pd.Series(VAL_RESULTS, name=f"Recall@{K}, валидация {SCHEME}").round(4).to_frame())
print(f"полнота пула v3: {analysis.pool_hit_rate(val_pool, val_s.n_rel).mean():.4f} | "
      f"в ответ идёт: {'ранкер' if USE_RANKER else 'формула v2'}")

In [ ]:
final_val_score = ranker_val_score if USE_RANKER else val_pool["stage1"].to_numpy(np.float64)
val_pred = predict_from_scores(val_pool, val_index, final_val_score, K, DEC, nq)
r50 = per_query_recall(val_pred, val_s.truth, K)
analysis.segment_table(val_s.queries, r50, analysis.pool_hit_rate(val_pool, val_s.n_rel))

In [ ]:
display(importance_table(RANKER).head(25))
errors = analysis.error_examples(val_pred, val_s.truth, val_s.queries, CORPORA[SCHEME][1],
                                 analysis.pool_hit_rate(val_pool, val_s.n_rel), n=15)
print(f"запросов без единого попадания: {int((r50 == 0).sum())} ({(r50 == 0).mean():.3f})")
errors

## 7. Предсказание для бенчмарка

Пул бенчмарка построен в разделе 3 по статистикам всего train.

In [ ]:
bench_score = ranker_score(RANKER, BENCH_POOL, N_THREADS) if USE_RANKER else BENCH_POOL["stage1"].to_numpy(np.float64)
bench_pred = predict_from_scores(BENCH_POOL, bench_index, bench_score, K, DEC, len(bench_q))

ANSWER_PATH = OUT_DIR / "answer.csv"
save_answer(bench_q["query_id"], bench_pred, ANSWER_PATH)
CHECK = validate_answer(ANSWER_PATH, bench_q["query_id"], bench_items["item_id"], K)
print(CHECK)
overlap_v2 = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, v2_pred)])
print(f"совпадение с ответом v2: {overlap_v2:.3f} объявлений из топ-50 в среднем")

## 8. Артефакты

Модель сохраняется в текстовом формате LightGBM; отчёт содержит метрики, конфиги и версии библиотек. Повторный запуск должен дать тот же md5 `answer.csv`.

In [ ]:
RANKER.save_model(str(WORK_DIR / f"ranker_{R.version}.txt"), num_iteration=RANKER.best_iteration)
report = {
    "version": R.version,
    "answer_md5": CHECK["md5"],
    "val_scheme": SCHEME,
    "calibration": calibration.reset_index().to_dict(orient="records"),
    "v2_reproduced": v2_md5 == R.v2_answer_md5,
    "val_recall@50": VAL_RESULTS,
    "fold_recall@50": fold_scores,
    "objective": OBJECTIVE,
    "best_iteration": RANKER.best_iteration,
    "use_ranker": bool(USE_RANKER),
    "config": CFG.as_dict(),
    "ranker_config": R.as_dict(),
    "versions": VERSIONS,
}
(WORK_DIR / f"report_{R.version}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str))

for name, value in VAL_RESULTS.items():
    print(f"{name:28s} {value:.4f}")
print(f"answer.csv md5: {CHECK['md5']}")